<a href="https://colab.research.google.com/github/pablonvsx/pisi3-ufrpe/blob/main/data-science/notebooks/ML/analise_final/svm_temporal_2000_2008_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SVM — Amostra Temporal 2000–2008

Este notebook aplica o algoritmo **Support Vector Machine (SVM)** sobre a amostra temporal `water_quality_2000_2008.parquet`, que contém registros entre 2000 e 2008.

## Variáveis de entrada

Foram selecionadas 5 variáveis físico-químicas com correspondência direta com os parâmetros da Resolução CONAMA nº 357/2005 e sem risco de vazamento de dados:

| Variável | Justificativa |
|---|---|
| `Dissolved Oxygen (mg/l)` | Indicador essencial da qualidade ecológica |
| `Biochemical Oxygen Demand (mg/l)` | Indicador de poluição orgânica |
| `pH (ph units)` | Parâmetro clássico de qualidade da água |
| `Nitrate (mg/l)` | Associado à presença de nutrientes e contaminação |
| `Ammonia (mg/l)` | Indicador de poluição nitrogenada |

> As colunas auxiliares `ph_ok`, `od_ok`, `dbo_ok`, `nitrate_ok` e `ammonia_ok` foram **excluídas** por terem sido usadas diretamente na construção do rótulo `conama_status`. Incluí-las causaria vazamento de dados (*data leakage*).

## Desbalanceamento das classes

| Classe | Registros | Proporção |
|---|---|---|
| Excelente | 41.169 | ~68,7% |
| Boa | 11.796 | ~19,7% |
| Atenção | 5.842 | ~9,7% |
| Crítica | 1.089 | ~1,8% |

Devido ao forte desbalanceamento, foram aplicadas quatro estratégias:

1. **Sem balanceamento** — baseline puro
2. **`class_weight='balanced'`** — o SVM ajusta automaticamente os pesos inversamente proporcionais à frequência
3. **Balanceamento manual** — pesos definidos explicitamente com base na proporção inversa de cada classe
4. **SMOTE** — geração sintética de amostras para as classes minoritárias antes do treinamento

# Preparação do Ambiente

In [1]:
# INSTALAÇÃO DO IMBALANCED-LEARN (necessário para SMOTE no Colab)
!pip install imbalanced-learn -q

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

warnings.filterwarnings("ignore")

SEED = 42

print("Ambiente configurado com sucesso.")

Ambiente configurado com sucesso.


In [3]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    DATA_PATH = "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/water_quality_2000_2008_novorotulo.parquet"
else:
    DATA_PATH = "../../dataset/processed/water_quality_2000_2008_novorotulo.parquet"

df = pd.read_parquet(DATA_PATH)

print("Dataset carregado com sucesso.")
print(f"Shape: {df.shape}")
print(f"\nDistribuição do rótulo:")
print(df["conama_status"].value_counts())

Mounted at /content/drive
Dataset carregado com sucesso.
Shape: (59896, 23)

Distribuição do rótulo:
conama_status
Adequada        41169
Boa             11796
Não adequada     6931
Name: count, dtype: int64


# Seleção e Preparação das Features

In [4]:
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# FEATURES SELECIONADAS
feature_cols = [
    "Temperature (cel)",
    "Orthophosphate (mg/l)",
    "Nitrogen (mg/l)",
    "Country",
    "Waterbody Type"
]

categorical_cols = ["Country", "Waterbody Type"]
numeric_cols = ["Temperature (cel)", "Orthophosphate (mg/l)", "Nitrogen (mg/l)"]

X = df[feature_cols].copy()

# CODIFICAÇÃO DO TARGET
le = LabelEncoder()
y = le.fit_transform(df["conama_status"])

print("Classes codificadas:")
for i, classe in enumerate(le.classes_):
    print(f"  {i} → {classe}")

print(f"\nShape do X: {X.shape}")
print(f"Shape do y: {y.shape}")
print(f"\nValores nulos por coluna:")
print(X.isnull().sum())

Classes codificadas:
  0 → Adequada
  1 → Boa
  2 → Não adequada

Shape do X: (59896, 5)
Shape do y: (59896,)

Valores nulos por coluna:
Temperature (cel)        0
Orthophosphate (mg/l)    0
Nitrogen (mg/l)          0
Country                  0
Waterbody Type           0
dtype: int64


In [5]:
# TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print(f"Treino: {X_train.shape}")
print(f"Teste:  {X_test.shape}")

Treino: (47916, 5)
Teste:  (11980, 5)


In [6]:
# PRÉ-PROCESSADOR
# - StandardScaler nas numéricas (SVM é sensível à escala)
# - OneHotEncoder nas categóricas
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_cols
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_cols
        )
    ]
)

# Análise Exploratória das Features

Antes dos experimentos, uma visualização rápida das 5 variáveis selecionadas com as linhas de referência da CONAMA Classe 2.

In [7]:
# DISTRIBUIÇÃO DAS FEATURES POR CLASSE
numeric_cols_plot = ["Temperature (cel)", "Orthophosphate (mg/l)", "Nitrogen (mg/l)"]

df_plot = df[numeric_cols_plot].copy()
df_plot["conama_status"] = df["conama_status"].values

for col in numeric_cols_plot:
    fig = px.box(
        df_plot,
        x="conama_status",
        y=col,
        color="conama_status",
        title=f"Distribuição de {col} por classe CONAMA",
        color_discrete_map={
            "Boa": "#2ecc71",
            "Adequada": "#f39c12",
            "Não adequada": "#e74c3c"
        },
        category_orders={"conama_status": ["Boa", "Adequada", "Não adequada"]}
    )
    fig.show()


# Experimento 1 — SVM sem Balanceamento

Baseline puro. O modelo é treinado sem qualquer estratégia de compensação para o desbalanceamento entre classes.

O SVM exige normalização das features porque é sensível à escala. Por isso, o `StandardScaler` é aplicado dentro do pipeline antes do classificador.

In [8]:
model_1 = Pipeline([
    ("preprocessor", preprocessor),
    ("svm", SVC(
        kernel="rbf",
        random_state=SEED
    ))
])

model_1.fit(X_train, y_train)

y_train_pred_1 = model_1.predict(X_train)
y_pred_1 = model_1.predict(X_test)

print("=" * 50)
print("EXPERIMENTO 1 — SEM BALANCEAMENTO")
print("=" * 50)

print(f"Train Accuracy: {accuracy_score(y_train, y_train_pred_1):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, y_pred_1):.4f}")

print("\nClassification Report — TREINO:")
print(classification_report(y_train, y_train_pred_1, target_names=le.classes_))

print("\nClassification Report — TESTE:")
print(classification_report(y_test, y_pred_1, target_names=le.classes_))

print("Confusion Matrix — TESTE:")
print(confusion_matrix(y_test, y_pred_1))

EXPERIMENTO 1 — SEM BALANCEAMENTO
Train Accuracy: 0.7301
Test Accuracy:  0.7188

Classification Report — TREINO:
              precision    recall  f1-score   support

    Adequada       0.78      0.93      0.85     32935
         Boa       0.51      0.21      0.30      9436
Não adequada       0.48      0.41      0.44      5545

    accuracy                           0.73     47916
   macro avg       0.59      0.52      0.53     47916
weighted avg       0.69      0.73      0.69     47916


Classification Report — TESTE:
              precision    recall  f1-score   support

    Adequada       0.78      0.92      0.84      8234
         Boa       0.47      0.20      0.28      2360
Não adequada       0.45      0.38      0.41      1386

    accuracy                           0.72     11980
   macro avg       0.57      0.50      0.51     11980
weighted avg       0.68      0.72      0.68     11980

Confusion Matrix — TESTE:
[[7602  312  320]
 [1554  476  330]
 [ 636  217  533]]


# Experimento 2 — SVM com `class_weight='balanced'`

O parâmetro `class_weight='balanced'` instrui o SVM a ajustar automaticamente os pesos de cada classe inversamente proporcional à sua frequência no conjunto de treino.

Isso penaliza mais os erros nas classes minoritárias (Atenção e Crítica), forçando o modelo a prestar mais atenção a elas durante o treinamento.

In [9]:
model_2 = Pipeline([
    ("preprocessor", preprocessor),
    ("svm", SVC(
        kernel="rbf",
        class_weight="balanced",
        random_state=SEED
    ))
])

model_2.fit(X_train, y_train)

y_train_pred_2 = model_2.predict(X_train)
y_pred_2 = model_2.predict(X_test)

print("=" * 50)
print("EXPERIMENTO 2 — class_weight='balanced'")
print("=" * 50)

print(f"Train Accuracy: {accuracy_score(y_train, y_train_pred_2):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, y_pred_2):.4f}")

print("\nClassification Report — TREINO:")
print(classification_report(y_train, y_train_pred_2, target_names=le.classes_))

print("\nClassification Report — TESTE:")
print(classification_report(y_test, y_pred_2, target_names=le.classes_))

print("Confusion Matrix — TESTE:")
print(confusion_matrix(y_test, y_pred_2))


EXPERIMENTO 2 — class_weight='balanced'
Train Accuracy: 0.6808
Test Accuracy:  0.6729

Classification Report — TREINO:
              precision    recall  f1-score   support

    Adequada       0.90      0.77      0.83     32935
         Boa       0.56      0.26      0.36      9436
Não adequada       0.31      0.87      0.46      5545

    accuracy                           0.68     47916
   macro avg       0.59      0.63      0.55     47916
weighted avg       0.76      0.68      0.69     47916


Classification Report — TESTE:
              precision    recall  f1-score   support

    Adequada       0.90      0.76      0.82      8234
         Boa       0.54      0.26      0.35      2360
Não adequada       0.30      0.84      0.44      1386

    accuracy                           0.67     11980
   macro avg       0.58      0.62      0.54     11980
weighted avg       0.76      0.67      0.69     11980

Confusion Matrix — TESTE:
[[6293  408 1533]
 [ 607  604 1149]
 [ 123   99 1164]]


# Experimento 3 — SVM com Balanceamento Manual

No balanceamento manual, os pesos são definidos explicitamente com base na proporção inversa de cada classe no conjunto de treino.

A fórmula utilizada é:

$$w_i = \frac{n\_total}{n\_classes \times n\_amostras_i}$$

Isso difere do `class_weight='balanced'` porque permite ajustar os pesos de forma controlada, possibilitando dar ainda mais ênfase a classes específicas se necessário.

In [10]:
# CÁLCULO DOS PESOS MANUAIS
classes, counts = np.unique(y_train, return_counts=True)
n_total = len(y_train)
n_classes = len(classes)

manual_weights = {
    int(c): round(n_total / (n_classes * n), 4)
    for c, n in zip(classes, counts)
}

print("Pesos manuais calculados:")
for k, v in manual_weights.items():
    print(f"  Classe {k} ({le.classes_[k]}): {v}")

Pesos manuais calculados:
  Classe 0 (Adequada): 0.485
  Classe 1 (Boa): 1.6927
  Classe 2 (Não adequada): 2.8804


In [11]:
model_3 = Pipeline([
    ("preprocessor", preprocessor),
    ("svm", SVC(
        kernel="rbf",
        class_weight=manual_weights,
        random_state=SEED
    ))
])

model_3.fit(X_train, y_train)

y_train_pred_3 = model_3.predict(X_train)
y_pred_3 = model_3.predict(X_test)

print("=" * 50)
print("EXPERIMENTO 3 — BALANCEAMENTO MANUAL")
print("=" * 50)
print(f"Train Accuracy: {accuracy_score(y_train, y_train_pred_3):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, y_pred_3):.4f}")
print("\nClassification Report — TREINO:")
print(classification_report(y_train, y_train_pred_3, target_names=le.classes_))
print("\nClassification Report — TESTE:")
print(classification_report(y_test, y_pred_3, target_names=le.classes_))
print("Confusion Matrix — TESTE:")
print(confusion_matrix(y_test, y_pred_3))


EXPERIMENTO 3 — BALANCEAMENTO MANUAL
Train Accuracy: 0.6808
Test Accuracy:  0.6729

Classification Report — TREINO:
              precision    recall  f1-score   support

    Adequada       0.90      0.77      0.83     32935
         Boa       0.56      0.26      0.36      9436
Não adequada       0.31      0.87      0.46      5545

    accuracy                           0.68     47916
   macro avg       0.59      0.63      0.55     47916
weighted avg       0.76      0.68      0.69     47916


Classification Report — TESTE:
              precision    recall  f1-score   support

    Adequada       0.90      0.76      0.82      8234
         Boa       0.54      0.26      0.35      2360
Não adequada       0.30      0.84      0.44      1386

    accuracy                           0.67     11980
   macro avg       0.58      0.62      0.54     11980
weighted avg       0.76      0.67      0.69     11980

Confusion Matrix — TESTE:
[[6293  408 1533]
 [ 607  604 1149]
 [ 123   99 1164]]


# Experimento 4 — SVM com SMOTE

O **SMOTE (Synthetic Minority Over-sampling Technique)** gera amostras sintéticas para as classes minoritárias antes do treinamento.

Diferente das abordagens anteriores que apenas ajustam os pesos, o SMOTE efetivamente aumenta o número de amostras das classes sub-representadas, criando novos exemplos sintéticos interpolando entre amostras existentes.

O SMOTE é aplicado **somente no conjunto de treino** para evitar contaminação do conjunto de teste. Por isso, é integrado ao pipeline via `ImbPipeline` da biblioteca `imbalanced-learn`.

In [ ]:
model_4 = ImbPipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(
        random_state=SEED
    )),
    ("svm", SVC(
        kernel="rbf",
        random_state=SEED
    ))
])

model_4.fit(X_train, y_train)

y_train_pred_4 = model_4.predict(X_train)
y_pred_4 = model_4.predict(X_test)

print("=" * 50)
print("EXPERIMENTO 4 — SMOTE")
print("=" * 50)

print(f"Train Accuracy: {accuracy_score(y_train, y_train_pred_4):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, y_pred_4):.4f}")

print("\nClassification Report — TREINO:")
print(classification_report(y_train, y_train_pred_4, target_names=le.classes_))

print("\nClassification Report — TESTE:")
print(classification_report(y_test, y_pred_4, target_names=le.classes_))

print("Confusion Matrix — TESTE:")
print(confusion_matrix(y_test, y_pred_4))

EXPERIMENTO 4 — SMOTE
Train Accuracy: 0.6374
Test Accuracy:  0.6321

Classification Report — TREINO:
              precision    recall  f1-score   support

     Atenção       0.29      0.46      0.35      4673
         Boa       0.56      0.25      0.35      9437
     Crítica       0.09      0.84      0.16       871
   Excelente       0.90      0.77      0.83     32935

    accuracy                           0.64     47916
   macro avg       0.46      0.58      0.42     47916
weighted avg       0.76      0.64      0.68     47916


Classification Report — TESTE:
              precision    recall  f1-score   support

     Atenção       0.27      0.43      0.33      1169
         Boa       0.56      0.25      0.34      2359
     Crítica       0.08      0.78      0.15       218
   Excelente       0.90      0.77      0.83      8234

    accuracy                           0.63     11980
   macro avg       0.45      0.56      0.41     11980
weighted avg       0.76      0.63      0.67     1198

# Resumo Comparativo

Consolidação dos resultados dos quatro experimentos para facilitar a comparação entre as estratégias de balanceamento.

In [ ]:
resumo = pd.DataFrame([
    {
        "Experimento": "1 — Sem balanceamento",
        "Estratégia": "Nenhuma",
        "Train Accuracy": round(accuracy_score(y_train, y_train_pred_1), 4),
        "Test Accuracy": round(accuracy_score(y_test, y_pred_1), 4),
        "Macro F1 (teste)": round(__import__('sklearn.metrics', fromlist=['f1_score']).f1_score(y_test, y_pred_1, average='macro'), 4)
    },
    {
        "Experimento": "2 — class_weight='balanced'",
        "Estratégia": "Peso automático inversamente proporcional",
        "Train Accuracy": round(accuracy_score(y_train, y_train_pred_2), 4),
        "Test Accuracy": round(accuracy_score(y_test, y_pred_2), 4),
        "Macro F1 (teste)": round(__import__('sklearn.metrics', fromlist=['f1_score']).f1_score(y_test, y_pred_2, average='macro'), 4)
    },
    {
        "Experimento": "3 — Balanceamento manual",
        "Estratégia": "Pesos definidos explicitamente via fórmula",
        "Train Accuracy": round(accuracy_score(y_train, y_train_pred_3), 4),
        "Test Accuracy": round(accuracy_score(y_test, y_pred_3), 4),
        "Macro F1 (teste)": round(__import__('sklearn.metrics', fromlist=['f1_score']).f1_score(y_test, y_pred_3, average='macro'), 4)
    },
    {
        "Experimento": "4 — SMOTE",
        "Estratégia": "Geração sintética de amostras minoritárias",
        "Train Accuracy": round(accuracy_score(y_train, y_train_pred_4), 4),
        "Test Accuracy": round(accuracy_score(y_test, y_pred_4), 4),
        "Macro F1 (teste)": round(__import__('sklearn.metrics', fromlist=['f1_score']).f1_score(y_test, y_pred_4, average='macro'), 4)
    },
    {
        "Experimento": "5 — GridSearch + balanced",
        "Estratégia": f"C={grid_search.best_params_['svm__C']}, gamma={grid_search.best_params_['svm__gamma']}, balanced",
        "Train Accuracy": round(accuracy_score(y_train, y_train_pred_5), 4),
        "Test Accuracy": round(accuracy_score(y_test, y_pred_5), 4),
        "Macro F1 (teste)": round(__import__('sklearn.metrics', fromlist=['f1_score']).f1_score(y_test, y_pred_5, average='macro'), 4)
    }
])

resumo


,Experimento,Estratégia,Train Accuracy,Test Accuracy
0,1 — Sem balanceamento,Nenhuma,0.9520,0.9455
1,2 — class_weight='balanced',Peso automático inversamente proporcional,0.9439,0.9401
2,3 — Balanceamento manual,Pesos definidos explicitamente via fórmula,0.9439,0.9401
3,4 — SMOTE,Geração sintética de amostras minoritárias,0.9416,0.9400


In [ ]:
# VISUALIZAÇÃO COMPARATIVA
resumo_melted = resumo.melt(
    id_vars="Experimento",
    value_vars=["Train Accuracy", "Test Accuracy"],
    var_name="Conjunto",
    value_name="Accuracy"
)

fig = px.bar(
    resumo_melted,
    x="Experimento",
    y="Accuracy",
    color="Conjunto",
    barmode="group",
    title="Comparativo de Accuracy — SVM com diferentes estratégias de balanceamento",
    text_auto=".4f",
    color_discrete_map={
        "Train Accuracy": "#3498db",
        "Test Accuracy": "#2ecc71"
    }
)
fig.update_layout(
    yaxis=dict(range=[0, 1]),
    xaxis_tickangle=-20
)
fig.show()